#### This preprocessing notebook aims to to continue on from the EDA notebooks and adjusting the finding to suit the model ready for training.

In [1]:
#Import necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import os

In [2]:
path = Path("/Users/stans/eportfolio/eportfolio_Stanley-Southwick/network-anomaly-detector/data/raw").glob("*.csv") # List all CSV files in the raw data directory

# Loop through the CSV to concatenate them into a single DataFrame
dataframes = []
for csv_file in path:
    df = pd.read_csv(csv_file)
    dataframes.append(df)
# Concatenate all DataFrames into one
network_dataset = pd.concat(dataframes, ignore_index=True)

# Strip whitespace from column names
network_dataset.columns = network_dataset.columns.str.strip()

 #### Dataset have now been loaded and concated I will now start working through the findings starting with dropping duplicate rows and columns after confirming the shape is right.

In [3]:
print(network_dataset.head())

   Destination Port  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0                22            166                  1                       1   
1             60148             83                  1                       2   
2               123          99947                  1                       1   
3               123          37017                  1                       1   
4                 0      111161336                147                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                            0                            0   
1                            0                            0   
2                           48                           48   
3                           48                           48   
4                            0                            0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                      0                      0            

In [4]:
print(network_dataset.shape)

(2830743, 79)


In [5]:
print(network_dataset.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Destination Port             int64  
 1   Flow Duration                int64  
 2   Total Fwd Packets            int64  
 3   Total Backward Packets       int64  
 4   Total Length of Fwd Packets  int64  
 5   Total Length of Bwd Packets  int64  
 6   Fwd Packet Length Max        int64  
 7   Fwd Packet Length Min        int64  
 8   Fwd Packet Length Mean       float64
 9   Fwd Packet Length Std        float64
 10  Bwd Packet Length Max        int64  
 11  Bwd Packet Length Min        int64  
 12  Bwd Packet Length Mean       float64
 13  Bwd Packet Length Std        float64
 14  Flow Bytes/s                 float64
 15  Flow Packets/s               float64
 16  Flow IAT Mean                float64
 17  Flow IAT Std                 float64
 18  Flow IAT Max                 int64  
 19  

In [6]:
#Drop duplicate column and duplicate rows
network_data = network_dataset.drop('Fwd Header Length.1', axis=1)
network_data = network_dataset.drop_duplicates()

#### Now to double check the removal of duplicates have worked

In [7]:
print(network_dataset.shape)
print('Fwd Header Length.1' in network_dataset.columns)

(2830743, 79)
True


#### The replacement of infinite values with NaN for further preprocessing

In [8]:
inf_before = np.isinf(network_dataset.select_dtypes(include=[float])).sum()
print("Columns with infinite values before replacement:")
print(inf_before[inf_before > 0])

network_data = network_dataset.replace([np.inf, -np.inf], np.nan)

is_infinite_float = np.isinf(network_dataset.select_dtypes(include=[float])).sum()
print("\nColumns with infinite values:")
print(is_infinite_float[is_infinite_float > 0])

Columns with infinite values before replacement:
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64

Columns with infinite values:
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


#### Now that we have changed the infinite values into NaN I will look into how much null values are in each column and decide on the strategy

In [9]:
# Check for null values
is_null = network_dataset.isna().sum()
print("\nColumns with null values:")
print(is_null[is_null > 0])

# Drop the null values
network_data = network_dataset.dropna()
# Check for null values again
is_null_after = network_dataset.isna().sum()
print("\nColumns with null values after dropping:")
print(is_null_after[is_null_after > 0])

print(network_dataset.shape)


Columns with null values:
Flow Bytes/s    1358
dtype: int64

Columns with null values after dropping:
Flow Bytes/s    1358
dtype: int64
(2830743, 79)
